# Longitudinal Tracking Experiment

This notebook simulates a continuous timeline of a user's device state to demonstrate how monitoring RTT over time can reveal behavioral patterns (e.g., waking up, going to sleep).

In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Add simulator to path
sys.path.append(os.path.abspath("../simulator"))

from receipt_generator import ReceiptGenerator
from device_state_model import DeviceState, DeviceStateModel, DEFAULT_DEVICE_STATES
from network_delay_model import NetworkDelayModel, WIFI, CELLULAR

# Use WiFi for clearer demonstrations of state changes
network_model = NetworkDelayModel(profile=WIFI, seed=2024)
device_model = DeviceStateModel(state_profiles=DEFAULT_DEVICE_STATES, seed=2024)
generator = ReceiptGenerator(device_model=device_model, network_model=network_model)

## 1. Scenario Definition

We define a timeline script: a list of states representing a user's behavior over 60 simulated minutes.

In [ ]:
# Timeline: (Duration in Minutes, State)
scenario_script = [
    (15, DeviceState.ACTIVE),   # User is chatting
    (10, DeviceState.IDLE),     # Put phone down (screen off)
    (20, DeviceState.DOZING),   # Deep sleep kicks in
    (5, DeviceState.ACTIVE),    # Wakes up briefly
    (10, DeviceState.IDLE)      # Phone down again
]

# Simulation settings
pings_per_minute = 4  # e.g., one message every 15 seconds

timeline_data = []
current_time = 0.0

for duration_mins, state in scenario_script:
    n_pings = duration_mins * pings_per_minute
    
    for _ in range(n_pings):
        rtt = generator.generate_receipt(state)
        
        # Store data even if None (Offline), though WiFi usually yields returns
        timeline_data.append({
            "Time_Min": current_time,
            "State": state.value,
            "RTT": rtt if rtt else np.nan
        })
        
        current_time += (1 / pings_per_minute)

df_time = pd.DataFrame(timeline_data)

## 2. Visualize Pulse Over Time

We plot the RTTs as a time series. Sudden jumps in RTT baseline indicate state transitions.

In [ ]:
plt.figure(figsize=(14, 6))

# Scatter plot for individual pings
sns.scatterplot(data=df_time, x="Time_Min", y="RTT", hue="State", palette="deep", s=30)

# Rolling average to show the trend line (Attackers would use this)
df_time["Rolling_RTT"] = df_time["RTT"].rolling(window=5, min_periods=1).mean()
sns.lineplot(data=df_time, x="Time_Min", y="Rolling_RTT", color="black", alpha=0.5, label="Rolling Avg (Trend)")

plt.title("Simulated User Timeline: Detecting Activity via RTT")
plt.xlabel("Time (Minutes)")
plt.ylabel("RTT (ms)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 3. Analysis

- **Step Changes**: The transition from Active (Low RTT) to Dozing (High RTT) is visibly distinct.
- **Dozing Jitter**: Note the high variance in the Dozing phase. This is characteristic of deep power-saving modes waking up periodically.
- **Inference**: An attacker observing this chart can deduce exactly when the user put their phone down and when they picked it back up.